Load prototype events

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

EVENT_FILE = (
    PROJECT_ROOT
    / "data"
    / "events"
    / "thermal_events_prototype.csv"
)

events = pd.read_csv(EVENT_FILE)

print("Rows:", len(events))
print("Columns:", events.columns.tolist())

events.head()

Rows: 439251
Columns: ['acq_date', 'lat_cell', 'lon_cell', 'event_number', 'start_time', 'end_time', 'observation_count', 'latitude', 'longitude', 'mean_frp', 'peak_frp', 'total_frp', 'mean_brightness', 'satellite', 'daynight', 'duration_minutes', 'persistent']


,acq_date,lat_cell,lon_cell,event_number,start_time,end_time,observation_count,latitude,longitude,mean_frp,peak_frp,total_frp,mean_brightness,satellite,daynight,duration_minutes,persistent
0,2025-01-01,997,7760,1,2025-01-01 08:01:00,2025-01-01 08:01:00,1,8.984920,77.60124,8.350,8.35,8.35,341.700,NOAA-20 VIIRS,D,0.0,False
1,2025-01-01,999,7761,1,2025-01-01 08:01:00,2025-01-01 08:01:00,1,9.000140,77.61005,2.680,2.68,2.68,330.680,NOAA-20 VIIRS,D,0.0,False
2,2025-01-01,1029,7804,1,2025-01-01 08:01:00,2025-01-01 08:01:00,1,9.278090,78.04165,1.580,1.58,1.58,330.240,NOAA-20 VIIRS,D,0.0,False
3,2025-01-01,1061,7763,1,2025-01-01 19:49:00,2025-01-01 19:49:00,2,9.562105,77.63444,1.105,1.23,2.21,303.515,NOAA-21 VIIRS,N,0.0,True
4,2025-01-01,1068,7647,1,2025-01-01 07:38:00,2025-01-01 07:38:00,1,9.630170,76.47209,2.110,2.11,2.11,335.260,Suomi-NPP VIIRS,D,0.0,False


Convert time columns

In [15]:
events["start_time"] = pd.to_datetime(
    events["start_time"],
    errors="coerce"
)

events["end_time"] = pd.to_datetime(
    events["end_time"],
    errors="coerce"
)

events["acq_date"] = pd.to_datetime(
    events["acq_date"],
    errors="coerce"
)

print(events[
    ["acq_date", "start_time", "end_time"]
].head())

    acq_date          start_time            end_time
0 2025-01-01 2025-01-01 08:01:00 2025-01-01 08:01:00
1 2025-01-01 2025-01-01 08:01:00 2025-01-01 08:01:00
2 2025-01-01 2025-01-01 08:01:00 2025-01-01 08:01:00
3 2025-01-01 2025-01-01 19:49:00 2025-01-01 19:49:00
4 2025-01-01 2025-01-01 07:38:00 2025-01-01 07:38:00


Basic event features

In [ ]:
events["duration_minutes"] = (
    events["end_time"] - events["start_time"]
).dt.total_seconds() / 60

events["frp_range"] = (
    events["peak_frp"] - events["mean_frp"]
)

thermal_features["frp_peak_ratio"] = np.where(
    thermal_features["mean_frp"] > 0,
    thermal_features["peak_frp"] / thermal_features["mean_frp"],
    0.0

)

events["frp_per_observation"] = (
    events["total_frp"] /
    events["observation_count"].replace(0, np.nan)
)

events.head()

,acq_date,lat_cell,lon_cell,event_number,start_time,end_time,observation_count,latitude,longitude,mean_frp,peak_frp,total_frp,mean_brightness,satellite,daynight,duration_minutes,persistent,frp_range,frp_peak_ratio,frp_per_observation
0,2025-01-01,997,7760,1,2025-01-01 08:01:00,2025-01-01 08:01:00,1,8.984920,77.60124,8.350,8.35,8.35,341.700,NOAA-20 VIIRS,D,0.0,False,0.000,1.000000,8.350
1,2025-01-01,999,7761,1,2025-01-01 08:01:00,2025-01-01 08:01:00,1,9.000140,77.61005,2.680,2.68,2.68,330.680,NOAA-20 VIIRS,D,0.0,False,0.000,1.000000,2.680
2,2025-01-01,1029,7804,1,2025-01-01 08:01:00,2025-01-01 08:01:00,1,9.278090,78.04165,1.580,1.58,1.58,330.240,NOAA-20 VIIRS,D,0.0,False,0.000,1.000000,1.580
3,2025-01-01,1061,7763,1,2025-01-01 19:49:00,2025-01-01 19:49:00,2,9.562105,77.63444,1.105,1.23,2.21,303.515,NOAA-21 VIIRS,N,0.0,True,0.125,1.113122,1.105
4,2025-01-01,1068,7647,1,2025-01-01 07:38:00,2025-01-01 07:38:00,1,9.630170,76.47209,2.110,2.11,2.11,335.260,Suomi-NPP VIIRS,D,0.0,False,0.000,1.000000,2.110


Temporal features

In [25]:
print(
    "Missing frp_peak_ratio:",
    thermal_features["frp_peak_ratio"].isna().sum()
)

print(
    "Infinite frp_peak_ratio:",
    np.isinf(thermal_features["frp_peak_ratio"]).sum()
)

NameError: name 'thermal_features' is not defined

In [17]:
events["start_hour"] = events["start_time"].dt.hour

events["start_minute"] = events["start_time"].dt.minute

events["day_of_week"] = (
    events["start_time"].dt.dayofweek
)

events["month"] = (
    events["start_time"].dt.month
)

events["is_weekend"] = (
    events["day_of_week"] >= 5
).astype(int)

events[
    [
        "start_hour",
        "day_of_week",
        "month",
        "is_weekend"
    ]
].head()

,start_hour,day_of_week,month,is_weekend
0,8,2,1,0
1,8,2,1,0
2,8,2,1,0
3,19,2,1,0
4,7,2,1,0


Persistence features

In [18]:
events["persistent"] = (
    events["observation_count"] >= 2
).astype(int)

events["long_duration"] = (
    events["duration_minutes"] >= 30
).astype(int)

events[
    [
        "observation_count",
        "duration_minutes",
        "persistent",
        "long_duration"
    ]
].head()

,observation_count,duration_minutes,persistent,long_duration
0,1,0.0,0,0
1,1,0.0,0,0
2,1,0.0,0,0
3,2,0.0,1,0
4,1,0.0,0,0


FRP categories

In [19]:
events["frp_level"] = pd.cut(
    events["peak_frp"],
    bins=[-np.inf, 5, 20, 50, 100, np.inf],
    labels=[
        "very_low",
        "low",
        "medium",
        "high",
        "very_high"
    ]
)

events["frp_level"].value_counts()

frp_level
very_low     263998
low          149070
medium        17409
high           4756
very_high      4018
Name: count, dtype: int64

Day/night features

In [20]:
events["is_day"] = (
    events["daynight"]
    .astype(str)
    .str.upper()
    .str.contains("D")
    .astype(int)
)

events["is_night"] = (
    events["daynight"]
    .astype(str)
    .str.upper()
    .str.contains("N")
    .astype(int)
)

events[
    ["daynight", "is_day", "is_night"]
].head()

,daynight,is_day,is_night
0,D,1,0
1,D,1,0
2,D,1,0
3,N,0,1
4,D,1,0


Check the feature table

In [21]:
print("Feature table shape:", events.shape)

display(events.head())

print("\nMissing values:")
display(
    events.isna().sum()
    .sort_values(ascending=False)
    .head(20)
)

Feature table shape: (439251, 29)


,acq_date,lat_cell,lon_cell,event_number,start_time,end_time,observation_count,latitude,longitude,mean_frp,...,frp_per_observation,start_hour,start_minute,day_of_week,month,is_weekend,long_duration,frp_level,is_day,is_night
0,2025-01-01,997,7760,1,2025-01-01 08:01:00,2025-01-01 08:01:00,1,8.984920,77.60124,8.350,...,8.350,8,1,2,1,0,0,low,1,0
1,2025-01-01,999,7761,1,2025-01-01 08:01:00,2025-01-01 08:01:00,1,9.000140,77.61005,2.680,...,2.680,8,1,2,1,0,0,very_low,1,0
2,2025-01-01,1029,7804,1,2025-01-01 08:01:00,2025-01-01 08:01:00,1,9.278090,78.04165,1.580,...,1.580,8,1,2,1,0,0,very_low,1,0
3,2025-01-01,1061,7763,1,2025-01-01 19:49:00,2025-01-01 19:49:00,2,9.562105,77.63444,1.105,...,1.105,19,49,2,1,0,0,very_low,0,1
4,2025-01-01,1068,7647,1,2025-01-01 07:38:00,2025-01-01 07:38:00,1,9.630170,76.47209,2.110,...,2.110,7,38,2,1,0,0,very_low,1,0



Missing values:


frp_peak_ratio         11
lat_cell                0
lon_cell                0
event_number            0
acq_date                0
end_time                0
observation_count       0
latitude                0
longitude               0
mean_frp                0
peak_frp                0
total_frp               0
start_time              0
mean_brightness         0
satellite               0
duration_minutes        0
daynight                0
persistent              0
frp_range               0
frp_per_observation     0
dtype: int64

elect prototype ML features

In [22]:
feature_columns = [
    "observation_count",
    "duration_minutes",
    "mean_frp",
    "peak_frp",
    "total_frp",
    "frp_range",
    "frp_peak_ratio",
    "frp_per_observation",
    "mean_brightness",
    "start_hour",
    "day_of_week",
    "month",
    "is_weekend",
    "persistent",
    "long_duration",
    "is_day",
    "is_night"
]

X = events[feature_columns].copy()

print("X shape:", X.shape)

display(X.head())

X shape: (439251, 17)


,observation_count,duration_minutes,mean_frp,peak_frp,total_frp,frp_range,frp_peak_ratio,frp_per_observation,mean_brightness,start_hour,day_of_week,month,is_weekend,persistent,long_duration,is_day,is_night
0,1,0.0,8.350,8.35,8.35,0.000,1.000000,8.350,341.700,8,2,1,0,0,0,1,0
1,1,0.0,2.680,2.68,2.68,0.000,1.000000,2.680,330.680,8,2,1,0,0,0,1,0
2,1,0.0,1.580,1.58,1.58,0.000,1.000000,1.580,330.240,8,2,1,0,0,0,1,0
3,2,0.0,1.105,1.23,2.21,0.125,1.113122,1.105,303.515,19,2,1,0,1,0,0,1
4,1,0.0,2.110,2.11,2.11,0.000,1.000000,2.110,335.260,7,2,1,0,0,0,1,0


Clean numeric features

In [23]:
X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.fillna(0)

print("Missing values:", X.isna().sum().sum())
print("Infinite values:", np.isinf(X).sum().sum())

Missing values: 0
Infinite values: 0


Save feature dataset

In [24]:
FEATURE_DIR = (
    PROJECT_ROOT
    / "data"
    / "features"
)

FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FEATURE_FILE = (
    FEATURE_DIR
    / "thermal_features_prototype.csv"
)

feature_data = events[
    [
        "acq_date",
        "start_time",
        "end_time",
        "latitude",
        "longitude"
    ] + feature_columns
].copy()

feature_data.to_csv(
    FEATURE_FILE,
    index=False
)

print("Saved:")
print(FEATURE_FILE)

Saved:
c:\Users\DIPANSHU SHUKLA\OneDrive\Desktop\SIH_162_ML\data\features\thermal_features_prototype.csv
